In [2]:
from resources.helper.helper_dataset import get_dataloaders_cifar10
from resources.helper.helper_evaluation import set_all_seeds, set_deterministic
from resources.helper.helper_train import train_model
from resources.helper.helper_plotting import plot_training_loss, plot_accuracy, show_examples

import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
set_all_seeds(42)

In [5]:
#no need to caluclate here already given to us
# resize_transform = torchvision.transforms.Compose(
#     [
#         torchvision.transforms.Resize((70, 70)),
#         torchvision.transforms.RandomCrop((64, 64)), 
#         torchvision.transforms.ToTensor()])
# train_loader, valid_loader, test_loader = get_dataloaders_cifar10(batch_size=256,validation_fraction=0.1, 
#                                                                 train_transforms=resize_transform,test_transforms=resize_transform)
# train_mean = []
# train_std = []

# for i, image in enumerate(train_loader, 0):
#     numpy_image = image[0].numpy()
    
#     batch_mean = np.mean(numpy_image, axis=(0, 2, 3))
#     batch_std = np.std(numpy_image, axis=(0, 2, 3))
    
#     train_mean.append(batch_mean)
#     train_std.append(batch_std)

# train_mean = np.mean(train_mean, axis=0)
# train_std = np.mean(train_std, axis=0)

# print('Mean:', train_mean)
# print('Std Dev:', train_std)

In [6]:
resize_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((70, 70)),
        torchvision.transforms.RandomCrop((64, 64)), 
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.485,0.456, 0.406), (0.229, 0.224, 0.225))])
test_resize_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((70, 70)),
        torchvision.transforms.CenterCrop((64, 64)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.485,0.456, 0.406), (0.229, 0.224, 0.225))]) #mean and std dev.

train_loader, valid_loader, test_loader = get_dataloaders_cifar10(batch_size=256,validation_fraction=0.1, 
                                                                train_transforms=resize_transform,test_transforms=resize_transform)

In [7]:
model = torchvision.models.vgg16(pretrained=True)
model

d:\Study\Deep learning\Code Practice\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Study\Deep learning\Code Practice\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [10]:
#freezing params
for param in model.parameters():
    param.requires_grad = False

In [11]:
#unfreezing params of weighted layers in classifier for training
model.classifier[0].requires_grad=True
model.classifier[3].requires_grad=True

#changing last layer according to our requirement (number of output classes)
model.classifier[6]=torch.nn.Linear(4096, 10)




In [12]:
model=model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,factor=0.1, mode='max')
Epochs=5
minibatch_loss_list, train_acc_list, valid_acc_list = train_model(
    model=model,
    num_epochs=Epochs,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    device=device,
    logging_interval=50,
    scheduler=scheduler,
    scheduler_on='valid_acc')

plot_training_loss(minibatch_loss_list=minibatch_loss_list,
                   num_epochs=Epochs,
                   iter_per_epoch=len(train_loader),
                   results_dir=None,
                   averaging_iterations=20)

plt.show()

plot_accuracy(train_acc_list=train_acc_list,
              valid_acc_list=valid_acc_list,
              results_dir=None)

plt.ylim([80, 100])
plt.show()

Epoch: 001/005 | Batch 0000/0175 | Loss: 2.6051
Epoch: 001/005 | Batch 0050/0175 | Loss: 9.6248


KeyboardInterrupt: 

In [ ]:
# torch.cuda.empty_cache()

NameError: name 'torch' is not defined